# 01. Data Exploration & Regime Detection

This notebook covers:
1. Downloading and aligning price data using `yfinance` with custom caching and cleaning.
2. Technical feature engineering with causal (non-lookahead) checks.
3. Market regime modeling using a 3-state Hidden Markov Model (HMM).
4. Transition matrix extraction and price chart regime-shading visualizations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_and_align_data, convert_to_log_returns
from src.features import build_features, standardize_features
from src.regime_model import RegimeHMM

## 1. Load and Clean Financial Data

In [ ]:
tickers = ["^NSEI", "GOLDBEES.NS", "LICNETFGSC.NS", "^INDIAVIX"]
start_date = "2010-01-01"
end_date = "2026-07-04"

# Download and align prices. This will auto-detect and fix the GOLDBEES 2019 bad tick anomaly.
prices = load_and_align_data(tickers, start_date, end_date, cache_dir="../data/raw")
log_returns = convert_to_log_returns(prices)

print(f"Aligned Price Dataset: {prices.shape[0]} trading days, from {prices.index.min().strftime('%Y-%m-%d')} to {prices.index.max().strftime('%Y-%m-%d')}")

## 2. Baseline Visualizations

In [ ]:
# Plot 1: Normalized Price Series (Rebased to 100 on first aligned day)
assets = ["^NSEI", "GOLDBEES.NS", "LICNETFGSC.NS"]
normalized = prices[assets].div(prices[assets].iloc[0]).mul(100)

plt.figure(figsize=(12, 6))
for col in assets:
    plt.plot(normalized.index, normalized[col], label=col, alpha=0.8, linewidth=1.5)
plt.title("Normalized Price Series (Rebased to 100)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot 2: Daily Log Returns
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for i, col in enumerate(assets):
    axes[i].plot(log_returns.index, log_returns[col], label=col, color=f"C{i}", alpha=0.6, linewidth=0.7)
    axes[i].set_title(f"Daily Log Returns - {col}", fontsize=12, fontweight="bold")
    axes[i].set_ylabel("Log Return")
    axes[i].legend(loc="upper left")
    axes[i].grid(True, alpha=0.3)
plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: India VIX with Crisis Spans
plt.figure(figsize=(12, 6))
plt.plot(prices.index, prices["^INDIAVIX"], label="India VIX", color="crimson", linewidth=1.2)
plt.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"), color="gray", alpha=0.25, label="2020 COVID Crisis")
plt.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31"), color="orange", alpha=0.15, label="2022 Macro Spikes")
plt.title("India VIX with Key Crisis Spans", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("VIX Level (%)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Feature Engineering

In [ ]:
# Build technical features for HMM (drops initial 199 NaN rows due to 200-day moving average)
features = build_features(prices)
print(f"Features shape: {features.shape}")

# Standardize features (Z-Score) fitted over the entire range for full history visualization
std_features = standardize_features(features, features.index.min().strftime('%Y-%m-%d'), features.index.max().strftime('%Y-%m-%d'))
std_features.head(3)

## 4. Hidden Markov Model (HMM) Regime Fitting

In [ ]:
# Fit 3-state HMM (using seed 100 for high stability and convergence)
hmm = RegimeHMM(n_regimes=3, random_state=100)
hmm.fit(std_features)

# Print transition matrix
trans_mat = hmm.get_transition_matrix()
print("Transition Probability Matrix:")
display(trans_mat.round(4))

## 5. Visualizing Regime Transitions on NIFTY

In [ ]:
labeled_regimes = hmm.predict_labeled_regimes(std_features)
nifty_prices = prices.loc[std_features.index, "^NSEI"]

plt.figure(figsize=(14, 8))
plt.plot(nifty_prices.index, nifty_prices.values, color="black", linewidth=1.5, label="NIFTY 50", alpha=0.9)

color_map = {"Bull": "green", "Bear": "orange", "Crisis": "red"}

# Shading spans by regime blocks
current_reg = labeled_regimes.iloc[0]
start_dt = labeled_regimes.index[0]

# Dummy legend items
plt.axvspan(start_dt, start_dt, color="green", alpha=0.15, label="Bull Regime")
plt.axvspan(start_dt, start_dt, color="orange", alpha=0.15, label="Bear Regime")
plt.axvspan(start_dt, start_dt, color="red", alpha=0.15, label="Crisis Regime")

for date, regime in labeled_regimes.items():
    if regime != current_reg:
        plt.axvspan(start_dt, date, color=color_map.get(current_reg, "gray"), alpha=0.15)
        current_reg = regime
        start_dt = date
plt.axvspan(start_dt, labeled_regimes.index[-1], color=color_map.get(current_reg, "gray"), alpha=0.15)

plt.title("NIFTY 50 Index with HMM Market Regimes Shading (Seed 100)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Index Level")
plt.yscale("log")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()